In [33]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from students.kiselev.lesson2 import Exercise


In [ ]:
iris = load_iris()
X = iris.data.astype(float)
y = iris.target.astype(int)

y_binary = (y == 0).astype(int)

print(f"размер {X.shape}")
print(f"позитивиные: {int(np.sum(y_binary == 1))}")
print(f"негативные: {int(np.sum(y_binary == 0))}")


размер (150, 4)
позитивиные: 50
негативные: 100
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


In [35]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_binary,
    test_size=0.4,
    random_state=42,
    stratify=y_binary,
    shuffle=True,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp,
    shuffle=True,
)

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Val:   {len(X_val)} ({len(X_val)/len(X)*100:.0f}%)")
print(f"Test:  {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")

print(f"Train: pos={int(np.sum(y_train==1))}, neg={int(np.sum(y_train==0))}")
print(f"Val:   pos={int(np.sum(y_val==1))}, neg={int(np.sum(y_val==0))}")
print(f"Test:  pos={int(np.sum(y_test==1))}, neg={int(np.sum(y_test==0))}")


Train: 90 (60%)
Val:   30 (20%)
Test:  30 (20%)
Train: pos=30, neg=60
Val:   pos=10, neg=20
Test:  pos=10, neg=20


In [36]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1.0

X_train_norm = (X_train - mean) / std
X_val_norm = (X_val - mean) / std
X_test_norm = (X_test - mean) / std

print("нормализация:")
print("mean:", mean)
print("std: ", std)


нормализация:
mean: [5.84444444 3.03555556 3.67666667 1.14444444]
std:  [0.78924874 0.43366939 1.66893512 0.71741993]


In [37]:
def evaluate_model(lr, batch_size, X_train, X_val, y_train, y_val, n_epochs=25, seed=42):
    model = Exercise.create_logistic_model(num_features=X_train.shape[1], rng=np.random.default_rng(seed))
    Exercise.fit(model, X_train, y_train, lr=lr, n_epoch=n_epochs, batch_size=batch_size)

    y_pred_proba = model.predict(X_val)
    y_pred = (y_pred_proba > 0.5).astype(int)

    accuracy = float(np.mean(y_pred == y_val))
    auroc = float(roc_auc_score(y_val, y_pred_proba))
    return accuracy, auroc


In [38]:
learning_rates = [
    0.001, 0.003, 0.004, 0.005, 0.006, 0.007, 0.009,
    0.01, 0.02, 0.03, 0.04, 0.05,
    0.06, 0.07, 0.08, 0.09, 0.1,
    0.12, 0.14, 0.16, 0.18, 0.2,
    0.22, 0.24, 0.26, 0.28, 0.3,
    0.35, 0.4, 0.45, 0.5, 0.55,
    0.6, 0.65, 0.7, 0.75, 0.8,
    0.85, 0.9, 0.95, 1.0,
]

batch_sizes = [4, 8, 16, 32, 64, None]
seeds = [1, 2, 3, 4, 5]

n_epochs = 25
results = []

for lr in learning_rates:
    for bs in batch_sizes:
        auroc_scores = []
        accuracy_scores = []

        for seed in seeds:
            acc, auc = evaluate_model(lr, bs, X_train_norm, X_val_norm, y_train, y_val, n_epochs=n_epochs, seed=seed)
            accuracy_scores.append(acc)
            auroc_scores.append(auc)

        results.append({
            "lr": float(lr),
            "batch_size": bs,
            "mean_auroc": float(np.mean(auroc_scores)),
            "std_auroc": float(np.std(auroc_scores)),
            "mean_accuracy": float(np.mean(accuracy_scores)),
            "std_accuracy": float(np.std(accuracy_scores)),
            "min_auroc": float(np.min(auroc_scores)),
            "max_auroc": float(np.max(auroc_scores)),
            "min_accuracy": float(np.min(accuracy_scores)),
            "max_accuracy": float(np.max(accuracy_scores)),
        })

df_results = pd.DataFrame(results)
df_results["stability_auroc"] = df_results["mean_auroc"] - df_results["std_auroc"]
df_results["stability_accuracy"] = df_results["mean_accuracy"] - df_results["std_accuracy"]

df_results_auroc = df_results.sort_values(["stability_auroc", "mean_auroc"], ascending=False)
df_results_accuracy = df_results.sort_values(["stability_accuracy", "mean_accuracy"], ascending=False)

print("ТОП-5 AUROC")
print(df_results_auroc[["lr", "batch_size", "mean_auroc", "std_auroc", "stability_auroc", "mean_accuracy"]].head(5).to_string(index=False))

print("\nТОП-5 ACCURACY")
print(df_results_accuracy[["lr", "batch_size", "mean_accuracy", "std_accuracy", "stability_accuracy", "mean_auroc"]].head(5).to_string(index=False))

best_auroc_row = df_results_auroc.iloc[0]
best_lr = float(best_auroc_row["lr"])
best_bs = best_auroc_row["batch_size"]
if best_bs is not None:
    best_bs = int(best_bs)

print("\nлучший AUROC")
print(f"lr={best_lr}, batch_size={best_bs}")


ТОП-5 AUROC
   lr  batch_size  mean_auroc  std_auroc  stability_auroc  mean_accuracy
0.005         4.0         1.0        0.0              1.0       1.000000
0.006         4.0         1.0        0.0              1.0       1.000000
0.007         4.0         1.0        0.0              1.0       1.000000
0.009         4.0         1.0        0.0              1.0       1.000000
0.009         8.0         1.0        0.0              1.0       0.993333

ТОП-5 ACCURACY
   lr  batch_size  mean_accuracy  std_accuracy  stability_accuracy  mean_auroc
0.005         4.0            1.0           0.0                 1.0         1.0
0.006         4.0            1.0           0.0                 1.0         1.0
0.007         4.0            1.0           0.0                 1.0         1.0
0.009         4.0            1.0           0.0                 1.0         1.0
0.010         4.0            1.0           0.0                 1.0         1.0

лучший AUROC
lr=0.005, batch_size=4


In [ ]:
X_trainval = np.vstack([X_train_norm, X_val_norm])
y_trainval = np.concatenate([y_train, y_val])

final_model = Exercise.create_logistic_model(num_features=X_trainval.shape[1], rng=np.random.default_rng(42))
Exercise.fit(final_model, X_trainval, y_trainval, lr=best_lr, n_epoch=25, batch_size=best_bs)

test_proba = final_model.predict(X_test_norm)
test_pred = (test_proba > 0.5).astype(int)

test_accuracy = float(np.mean(test_pred == y_test))
test_auroc = float(roc_auc_score(y_test, test_proba))

print(f"best lr={best_lr}, batch_size={best_bs}")
print(f"test accuracy={test_accuracy:.4f}")
print(f"test AUROC={test_auroc:.4f}")



FINAL TEST RESULTS
best lr=0.005, batch_size=4
test accuracy=0.9667
test AUROC=1.0000
